In [7]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import os

def run_pipeline(input_file):
    try:
        # Load with latin1 to handle the SuperStore encoding issues
        df = pd.read_csv(input_file, encoding='latin1')
        
        # Standardize column names
        df.columns = [c.replace(' ', '_').replace('-', '_').lower() for c in df.columns]
        
        # Financial Engineering: Profit Margin %
        df['margin_pct'] = (df['profit'] / df['sales']) * 100
        
        # Behavioral Engineering: 'Toxic' Orders (Discount > 20% & Negative Profit)
        df['leakage_flag'] = np.where((df['discount'] > 0.2) & (df['profit'] < 0), 1, 0)

        # Connect to your local PostgreSQL
        engine = create_engine('postgresql://shreyd04@localhost:5432/sales_db')
        df.to_sql('orders', engine, if_exists='replace', index=False)
        
        # --- PATH FIX FOR STREAMLIT ---
        # This ensures the CSV is saved in the root folder, not inside the scripts folder
        project_root = "/Users/shreyd04/SuperStore Sales Dashboard"
        output_csv = os.path.join(project_root, 'processed_sales_risk.csv')
        
        df.to_csv(output_csv, index=False)
        print(f"✅ Pipeline Complete: Data migrated and CSV saved at {output_csv}")

    except Exception as e:
        print(f"❌ Pipeline failed: {e}")

if __name__ == "__main__":
    path = '/Users/shreyd04/SuperStore Sales Dashboard/data/samplesuperstore.csv'
    run_pipeline(path)

✅ Pipeline Complete: Data migrated and CSV saved at /Users/shreyd04/SuperStore Sales Dashboard/processed_sales_risk.csv
